# Employee Data Cleaning with Pandas

This project demonstrates a full data cleaning workflow using Pandas on a messy HR dataset.

In [ ]:
import pandas as pd
import numpy as np
import sys
from datetime import datetime
import pytz

In [ ]:
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    path = '/content/drive/MyDrive/google_colab/dirty_dataset.csv'
else:
    path = 'dirty_dataset.csv'

df = pd.read_csv(path, sep=";")

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.describe()

In [ ]:
df.value_counts()

In [ ]:
df_col_rename = df.copy()
df_col_rename.rename(columns={"Tööala": "Amet", "Liitumise_Kuupäev": "Liitumise_kuupäev"}, inplace=True)


In [ ]:
df_col_rename.head()

In [ ]:
df_name = df_col_rename.copy()


In [ ]:
df_name[["Eesnimi", "Perekonnanimi"]] = df_name["Nimi1"].str.split(" ", n=1, expand=True)


In [ ]:
df_name["Eesnimi"] = df_name["Eesnimi"].str.strip()
df_name["Eesnimi"] = df_name["Eesnimi"].str.title()

In [ ]:
df_name["Perekonnanimi"] = df_name["Perekonnanimi"].str.strip()
df_name["Perekonnanimi"] = df_name["Perekonnanimi"].str.title()

In [ ]:
df_name = df_name.drop(columns=["Nimi1"])

In [ ]:
df_name.head()

In [ ]:
df_age = df_name.copy()

In [ ]:
df_age["Vanus"] = df_age["Vanus"].astype(str).str.extract(r'(\d+)')
df_age["Vanus"] = pd.to_numeric(df_age["Vanus"], errors="coerce")

In [ ]:
df_age.loc[(df_age["Vanus"] < 15) | (df_age["Vanus"] > 100), "Vanus"] = None

In [ ]:
df_age["Vanus"] = df_age["Vanus"].fillna(df_age["Vanus"].median())

In [ ]:
df_age["Vanus"] = df_age["Vanus"].astype(int)

In [ ]:
df_age = df_age[(df_age["Vanus"] >= 15) & (df_age["Vanus"] <= 100)]

In [ ]:
df_age["Vanus"].median()

In [ ]:
df_age["Vanus"].unique()

In [ ]:
df_age.head()

In [ ]:
df_date = df_age.copy()

In [ ]:
df_date["Liitumise_kuupäev"] = pd.to_datetime(
    df_date["Liitumise_kuupäev"].str.replace(".", "-", regex=False).str.replace("/", "-", regex=False),
    errors='coerce'
)

In [ ]:
df_date["Liitumise_kuupäev"] = df_date["Liitumise_kuupäev"].fillna(
    df_date["Liitumise_kuupäev"].median()
)

In [ ]:
print(df_age.columns.tolist())

In [ ]:
df_date

In [ ]:
df_salary = df_date.copy()

In [ ]:
df_salary["Palk"] = pd.to_numeric(df_salary["Palk"], errors="coerce")

In [ ]:
df_salary["Palk"] = df_salary["Palk"].fillna(
    round(df_salary["Palk"].mean())
)

In [ ]:
df_salary["Palk"] = df_salary["Palk"].astype(int)

In [ ]:
df_salary.head()

In [ ]:
df_dropped = df_salary.copy()

In [ ]:
if "Parkimine" in df_dropped.columns:
    df_dropped.drop(columns=["Parkimine"], inplace=True)

In [ ]:
df_dropped.head()

In [ ]:
df_title = df_dropped.copy()

In [ ]:
df_title["Amet"] = df_title["Amet"].replace({
    "Finance": "FI",
    "Human Resources": "HR",
    "Ops": "OP",

})

In [ ]:
mode_amet = df_title["Amet"].mode()[0]
df_title["Amet"] = df_title["Amet"].fillna(mode_amet)

In [ ]:
df_title["Amet"].unique()

In [ ]:
df_title

In [ ]:
df_dupl = df_title.copy()

In [ ]:
df_dupl["Eesnimi"] = df_dupl["Eesnimi"].str.strip().str.capitalize()
df_dupl["Perekonnanimi"] = df_dupl["Perekonnanimi"].str.strip().str.title()

In [ ]:
df_dupl = df_dupl.drop_duplicates(
    subset=["Eesnimi","Perekonnanimi"],
    keep="first"
)


In [ ]:
print("df_title rows:", len(df_title))
print("df_dupl rows:", len(df_dupl))

In [ ]:
len(df_dupl)

In [ ]:
print(df_dupl.index)

In [ ]:
df_dt = df_dupl.copy()

In [ ]:
from datetime import datetime

In [ ]:
df_dt["Aastat_liitumisest"] = df_dt["Liitumise_kuupäev"].apply(
    lambda x: (datetime.now() - x).days // 365 if pd.notnull(x) else None
)

In [ ]:
df_dt

In [ ]:
df_cat = df_dt.copy()

In [ ]:
def palk_kategooria(palk):
    if pd.isna(palk):
        return np.nan
    elif 0 <= palk < 30000:
        return "madal"
    elif 30000 <= palk <= 59999:
        return "keskmine"
    elif palk >= 60000:
        return "kõrge"
    else:
        return np.nan

In [ ]:
df_cat["Palk_kategooria"] = df_cat["Palk"].apply(palk_kategooria)

In [ ]:
df_cat

In [ ]:
df_hoone = pd.DataFrame({
    "Id": [124,152,632,853,963,84,863,973,111,142],
    "Hoone": ["A","B","C","A","C","B","A","A","B","C"]
})

In [ ]:
df_merged = pd.merge(df_cat, df_hoone, on="Id")

In [ ]:
df_merged

In [ ]:
df_sort = df_merged.copy()

In [ ]:
df_sort = df_sort.sort_values("Liitumise_kuupäev", ascending=False).reset_index(drop=True)

In [ ]:
df_sort

In [ ]:
df_col_sorted = df_sort.copy()

In [ ]:
final_columns = ["Id","Liitumise_kuupäev","Aastat_liitumisest","Amet","Eesnimi","Perekonnanimi",
                 "Palk","Palk_kategooria","Vanus"]

In [ ]:
df_col_sorted = df_col_sorted[final_columns]

In [ ]:
df_col_sorted

In [ ]:
from datetime import datetime
import pytz


est_timezone = pytz.timezone('Europe/Tallinn')

# Get the current time in Estonian timezone and format it to include time
timestamp = datetime.now(est_timezone).strftime("%Y%m%d_%H%M%S")

if "google.colab" in sys.modules:
    path = f"/content/drive/MyDrive/google_colab/clean_dataset_{timestamp}.csv"
else:
    path = f"clean_dataset_{timestamp}.csv"

df_col_sorted.to_csv(path, index=False)